# Qwen2.5-VL: Bounding Box Grounding on Satellite Imagery

**Goal:** Test whether Qwen2.5-VL can **localize** hydraulic features with bounding boxes on **satellite imagery** instead of DEM-only renderings.

**Context:** In attempt1, Qwen2.5-VL produced bounding boxes on DEM hillshade but often defaulted to whole-image bboxes for abstract features. With satellite imagery (in-distribution), we expect much tighter, more accurate bounding boxes.

**Roadmap reference:** Stage 6.2 — Qwen2.5-VL Grounding on Satellite

## Tasks
1. Run bbox grounding prompts on satellite RGB
2. Test: channel detection, road detection, vegetation boundaries
3. Compare bbox quality to attempt1 DEM results
4. If bboxes are localized, these feed into SAM (Stage 7)
5. Save timestamped outputs to `data/output/model-outputs/attempt2/qwen25vl/`

In [ ]:
import os
import base64
import io
import re
import json
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import rasterio
from PIL import Image
from openai import OpenAI

# Paths
DEM_PATH = Path("../../data/input/1m elevation.tif")
SATELLITE_PATH = Path("../../data/output/cimarron_satellite.png")
HILLSHADE_PATH = Path("../../data/output/cimarron_hillshade.png")
OUTPUT_DIR = Path("../../data/output/model-outputs/attempt2/qwen25vl")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Timestamp for this run
RUN_TS = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

print(f"DEM: {DEM_PATH}")
print(f"Satellite image: {SATELLITE_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Run timestamp: {RUN_TS}")

## 1.1 Connect to Qwen2.5-VL via OpenRouter

In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not OPENROUTER_API_KEY:
    raise ValueError(
        "Set OPENROUTER_API_KEY in your .env file or environment.\n"
        "Get a free key at: https://openrouter.ai/keys"
    )

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

MODEL = "qwen/qwen2.5-vl-72b-instruct"
QWEN_AVAILABLE = False

try:
    test = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Say hello in one word."}],
        max_tokens=10,
    )
    print(f"Qwen2.5-VL connection OK: {test.choices[0].message.content}")
    print(f"Model: {MODEL}")
    QWEN_AVAILABLE = True
except Exception as e:
    print(f"Qwen2.5-VL is NOT available.")
    print(f"Error: {e}")
    print("The rest of this notebook will be skipped.")

## 1.2 Load Images & DEM Metadata

In [ ]:
# Load DEM metadata for geo-coordinate conversion
with rasterio.open(DEM_PATH) as src:
    dem_transform = src.transform
    dem_bounds = src.bounds
    dem_crs = src.crs

print(f"CRS: {dem_crs}")
print(f"Bounds: {dem_bounds}")

# Load satellite image
if not SATELLITE_PATH.exists():
    raise FileNotFoundError(
        f"Satellite image not found at {SATELLITE_PATH}.\n"
        "Run the satellite acquisition notebook (Stage 5) first."
    )

img_satellite = Image.open(SATELLITE_PATH)
print(f"Satellite image: {img_satellite.size} ({img_satellite.mode})")

# Load hillshade for comparison
img_hillshade = None
if HILLSHADE_PATH.exists():
    img_hillshade = Image.open(HILLSHADE_PATH)
    print(f"Hillshade image: {img_hillshade.size} ({img_hillshade.mode})")

## 1.3 Inference & Coordinate Parsing Utilities

In [ ]:
def resize_for_api(img, max_width=1008):
    """Resize image for API, aligning to multiples of 28 (Qwen2.5-VL internal grid)."""
    if img.width > max_width:
        ratio = max_width / img.width
        new_w = (int(img.width * ratio) // 28) * 28
        new_h = (int(img.height * ratio) // 28) * 28
        return img.resize((new_w, new_h), Image.LANCZOS)
    new_w = (img.width // 28) * 28
    new_h = (img.height // 28) * 28
    if (new_w, new_h) != img.size:
        return img.resize((new_w, new_h), Image.LANCZOS)
    return img


def image_to_base64(img):
    """Convert PIL Image to base64 data URL."""
    img = resize_for_api(img)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{b64}"


def vision_query(image, prompt, max_tokens=512):
    """Send an image + text prompt to the vision model."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": image_to_base64(image)}},
                ],
            }
        ],
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


def parse_qwen_grounding(text, image_w, image_h):
    """Parse Qwen2.5-VL grounding output into bboxes and points."""
    detections = []

    json_patterns = [
        r'```json\s*([\s\S]*?)```',
        r'```\s*([\s\S]*?)```',
        r'(\[\s*\{[\s\S]*?\}\s*\])',
        r'(\{[^{}]*"(?:bbox_2d|point_2d)"[^{}]*\})',
    ]

    json_str = None
    for pattern in json_patterns:
        match = re.search(pattern, text)
        if match:
            json_str = match.group(1).strip()
            break

    if not json_str:
        json_str = text.strip()

    try:
        data = json.loads(json_str)
    except json.JSONDecodeError:
        for line in text.split("\n"):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    data = [obj]
                    break
            except json.JSONDecodeError:
                continue
        else:
            return detections

    if isinstance(data, dict):
        data = [data]

    for item in data:
        if not isinstance(item, dict):
            continue
        label = item.get("label", item.get("description", "detection"))

        if "bbox_2d" in item:
            coords = item["bbox_2d"]
            if isinstance(coords, list) and len(coords) == 4:
                x1, y1, x2, y2 = [float(c) for c in coords]
                bbox_w = x2 - x1
                bbox_h = y2 - y1
                area_ratio = (bbox_w * bbox_h) / (image_w * image_h)
                if area_ratio > 0.9:
                    continue  # Skip whole-image bboxes
                detections.append({"type": "bbox", "coords": [x1, y1, x2, y2], "label": label})

        if "point_2d" in item:
            coords = item["point_2d"]
            if isinstance(coords, list) and len(coords) == 2:
                detections.append({"type": "point", "coords": [float(c) for c in coords], "label": label})

    return detections


def qwen_ground(image, prompt):
    """Run a grounding query and parse the results."""
    resized = resize_for_api(image)
    scale_x = image.width / resized.width
    scale_y = image.height / resized.height

    raw = vision_query(image, prompt)
    detections = parse_qwen_grounding(raw, resized.width, resized.height)

    bboxes = []
    points = []
    geo_points = []

    for det in detections:
        if det["type"] == "bbox":
            x1, y1, x2, y2 = det["coords"]
            bboxes.append({
                "coords": [x1 * scale_x, y1 * scale_y, x2 * scale_x, y2 * scale_y],
                "label": det["label"],
            })
            cx = (x1 * scale_x + x2 * scale_x) / 2
            cy = (y1 * scale_y + y2 * scale_y) / 2
            geo_points.append((cx, cy, det["label"]))
        elif det["type"] == "point":
            x, y = det["coords"]
            points.append({"coords": [x * scale_x, y * scale_y], "label": det["label"]})
            geo_points.append((x * scale_x, y * scale_y, det["label"]))

    geo_coords = []
    if geo_points:
        for x_px, y_px, label in geo_points:
            easting, northing = rasterio.transform.xy(dem_transform, int(y_px), int(x_px))
            geo_coords.append((easting, northing, label))

    return {
        "raw_output": raw,
        "bboxes": bboxes,
        "points": points,
        "geo_coords": geo_coords,
        "n_detections": len(detections),
    }


def plot_bboxes_on_image(img, bboxes, points, title, save_path=None, figsize=(14, 8)):
    """Overlay bounding boxes and points on an image."""
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(img)
    ax.set_title(title, fontsize=14)

    colors = plt.cm.Set1(np.linspace(0, 1, max(len(bboxes) + len(points), 1)))

    for i, bbox in enumerate(bboxes):
        x1, y1, x2, y2 = bbox["coords"]
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=colors[i], facecolor="none",
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, bbox["label"], fontsize=8, color=colors[i],
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

    for j, pt in enumerate(points):
        x, y = pt["coords"]
        ci = len(bboxes) + j
        ax.scatter([x], [y], c=[colors[ci]], s=100, marker="x", linewidths=2, zorder=5)
        ax.annotate(pt["label"], (x, y), textcoords="offset points",
                    xytext=(5, 5), fontsize=8, color=colors[ci],
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

    n_total = len(bboxes) + len(points)
    ax.set_xlabel(f"{len(bboxes)} bboxes, {len(points)} points" if n_total > 0
                  else "No detections parsed")
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()
    return fig


grounding_results = {}
print("Utilities ready.")

## 2.1 Grounding Tests — Satellite

In [ ]:
if not QWEN_AVAILABLE:
    print("Skipping — Qwen2.5-VL is not available.")
else:
    satellite_grounding_prompts = {
        "channel": (
            "Detect the river channel in this satellite image. "
            "Output a JSON array where each object has "
            '{"bbox_2d": [x1, y1, x2, y2], "label": "description"}. '
            "Coordinates should be pixel positions. Return multiple tight bounding boxes "
            "for different segments of the channel."
        ),
        "roads": (
            "Detect all roads and road embankments in this satellite image. "
            "Output a JSON array where each object has "
            '{"bbox_2d": [x1, y1, x2, y2], "label": "description"}. '
            "Coordinates should be pixel positions."
        ),
        "vegetation_zones": (
            "Detect distinct vegetation zones in this satellite image: forests, grasslands, "
            "and agricultural fields. Output a JSON array where each object has "
            '{"bbox_2d": [x1, y1, x2, y2], "label": "description"}. '
            "Coordinates should be pixel positions. Return separate boxes for each zone."
        ),
        "sandbars": (
            "Detect any sand bars, point bars, or exposed sediment in this satellite image. "
            "Output a JSON array where each object has "
            '{"bbox_2d": [x1, y1, x2, y2], "label": "description"}. '
            "Coordinates should be pixel positions."
        ),
        "bridges": (
            "Detect any bridges, crossings, or structures over the river in this satellite image. "
            "Output a JSON array where each object has "
            '{"bbox_2d": [x1, y1, x2, y2], "label": "description"}. '
            "Coordinates should be pixel positions."
        ),
    }

    for key, prompt in satellite_grounding_prompts.items():
        test_name = f"satellite_{key}"
        print(f"\n{'='*80}")
        print(f"TEST: {test_name}")
        print(f"{'='*80}")

        result = qwen_ground(img_satellite, prompt)
        grounding_results[test_name] = {"prompt": prompt, **result}

        print(f"\nRaw output:\n{result['raw_output']}")
        print(f"\nDetections: {result['n_detections']} ({len(result['bboxes'])} bboxes, {len(result['points'])} points)")
        for bbox in result["bboxes"]:
            x1, y1, x2, y2 = bbox["coords"]
            print(f"  BBOX: ({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}) — {bbox['label']}")

        save_path = OUTPUT_DIR / f"{RUN_TS}_satellite_{key}.png"
        plot_bboxes_on_image(img_satellite, result["bboxes"], result["points"],
                             f"Qwen2.5-VL Grounding: {key} (satellite)",
                             save_path=save_path)

## 3. Results Summary & Save

In [ ]:
if not QWEN_AVAILABLE:
    print("Qwen2.5-VL was not available — no results to summarize.")
else:
    # Grounding results table
    print("GROUNDING RESULTS")
    print(f"{'Test':<30} {'BBoxes':>7} {'Points':>7}  {'Labels'}")
    print("-" * 80)
    for key, result in grounding_results.items():
        nb = len(result["bboxes"])
        np_ = len(result["points"])
        labels = ", ".join(
            [b["label"] for b in result["bboxes"]] +
            [p["label"] for p in result["points"]]
        )
        if not labels:
            labels = "(no detections parsed)"
        print(f"{key:<30} {nb:>7} {np_:>7}  {labels[:60]}")

    print(f"\nModel: {MODEL}")
    print(f"Total grounding tests: {len(grounding_results)}")

    # Save results to markdown
    md_lines = [
        f"# Qwen2.5-VL — Satellite Grounding Test Results\n",
        f"\n**Date:** {RUN_TS}\n",
        f"\n**Model:** `{MODEL}` (via OpenRouter)\n",
        f"\n**Input:** Satellite (NAIP RGB, 0.6m/pixel)\n",
        f"\n**DEM:** Cimarron River, 1m resolution, {dem_crs}\n",
        f"\n**Notebook:** `notebooks/attempt2/03_qwen25vl_satellite.ipynb`\n",
        f"\n**Attempt:** 2 (satellite imagery)\n",
        "\n---\n",
    ]

    for key, result in grounding_results.items():
        md_lines.append(f"\n## Grounding: {key.replace('_', ' ').title()}\n")
        md_lines.append(f"\n**Prompt:** {result['prompt']}\n")
        md_lines.append(f"\n**Detections:** {result['n_detections']} ({len(result['bboxes'])} bboxes, {len(result['points'])} points)\n")
        if result["bboxes"]:
            md_lines.append("\n**Bounding boxes (original pixel coords):**\n\n")
            for bbox in result["bboxes"]:
                x1, y1, x2, y2 = bbox["coords"]
                md_lines.append(f"- ({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}) — {bbox['label']}\n")
        if result["geo_coords"]:
            md_lines.append(f"\n**Geo-coordinates ({dem_crs}):**\n\n")
            for e, n, label in result["geo_coords"]:
                md_lines.append(f"- ({e:.1f}, {n:.1f}) — {label}\n")
        md_lines.append(f"\n**Raw output:**\n\n```\n{result['raw_output']}\n```\n")
        md_lines.append("\n---\n")

    # Summary table
    md_lines.append("\n## Summary\n\n")
    md_lines.append("| Test | BBoxes | Points | Notes |\n")
    md_lines.append("|------|--------|--------|-------|\n")
    for key, result in grounding_results.items():
        nb = len(result["bboxes"])
        np_ = len(result["points"])
        note = "detections parsed" if (nb + np_) > 0 else "no detections"
        md_lines.append(f"| {key} | {nb} | {np_} | {note} |\n")

    output_path = OUTPUT_DIR / f"{RUN_TS}_qwen25vl_satellite_results.md"
    with open(output_path, "w") as f:
        f.writelines(md_lines)

    print(f"\nResults saved to: {output_path}")